In [2]:
import os
os.makedirs('/content/models/response_generator', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/logs', exist_ok=True)
print("✅ Folders ready — let's go!")

✅ Folders ready — let's go!


In [3]:
import os
for folder in ['/content/models/response_generator', '/content/data', '/content/logs']:
    print(f"✅ {folder} exists: {os.path.exists(folder)}")

✅ /content/models/response_generator exists: True
✅ /content/data exists: True
✅ /content/logs exists: True


In [4]:
!pip install transformers datasets torch sentencepiece evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00


In [5]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

model_name = "google/flan-t5-base"

print("Loading tokenizer...")
tokenizer = T5Tokenizer.from_pretrained(model_name)

print("Loading model...")
model = T5ForConditionalGeneration.from_pretrained(model_name)

print("✅ Flan-T5 loaded successfully!")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Flan-T5 loaded successfully!


In [6]:
training_data = [

    # ── CHECK BALANCE ──────────────────────────────────────────
    {"input": "intent: check-balance | entities: account=chequing | user: What is my chequing balance?",
     "output": "Your chequing account balance is available in the mobile app or at any ATM. Is there anything else I can help you with?"},

    {"input": "intent: check-balance | entities: account=savings | user: How much do I have in savings?",
     "output": "Your savings account balance is available in the mobile app. Would you like to do anything else?"},

    {"input": "intent: check-balance | entities: account=all | user: Show me all my account balances",
     "output": "You can view all your account balances by logging into the mobile app or visiting any branch. Shall I help you with anything else?"},

    # ── TRANSFER FUNDS ─────────────────────────────────────────
    {"input": "intent: transfer-funds | entities: amount=200, from=chequing, to=savings | user: Transfer $200 from chequing to savings",
     "output": "I have received your request to transfer $200 from your chequing to your savings account. Please confirm to proceed."},

    {"input": "intent: transfer-funds | entities: amount=500, from=savings, to=chequing | user: Move $500 to my chequing account",
     "output": "Your request to transfer $500 from savings to chequing is ready. Please confirm to complete the transfer."},

    {"input": "intent: transfer-funds | entities: amount=1000, from=chequing, to=John | user: Send $1000 to John",
     "output": "I have noted your request to send $1000 to John. Please confirm the recipient details and approve to proceed."},

    # ── PAY BILL ───────────────────────────────────────────────
    {"input": "intent: pay-bill | entities: payee=Rogers, amount=120 | user: Pay my Rogers bill of $120",
     "output": "Your request to pay $120 to Rogers is ready. Please confirm to complete the payment."},

    {"input": "intent: pay-bill | entities: payee=Hydro, amount=85 | user: Pay my hydro bill",
     "output": "Your request to pay $85 to Hydro has been received. Please confirm to proceed with the payment."},

    {"input": "intent: pay-bill | entities: payee=Bell, amount=95 | user: I need to pay my Bell bill",
     "output": "Your request to pay $95 to Bell is ready. Please confirm to complete the payment."},

    # ── LOAN INQUIRY ───────────────────────────────────────────
    {"input": "intent: loan-inquiry | entities: loan_type=mortgage | context: Mortgage rates start at 4.5% | user: What are your mortgage rates?",
     "output": "Our current mortgage rates start at 4.5%. Would you like to speak with a loan advisor for a personalized quote?"},

    {"input": "intent: loan-inquiry | entities: loan_type=personal | context: Personal loans available from 6.9% | user: How much can I borrow for a personal loan?",
     "output": "Personal loans are available starting at 6.9% APR. The amount depends on your credit profile. Would you like to speak with an advisor?"},

    {"input": "intent: loan-inquiry | entities: loan_type=car | context: Auto loans from 5.2% | user: Do you offer car loans?",
     "output": "Yes, we offer auto loans starting at 5.2%. Would you like to book an appointment with a loan specialist?"},

    # ── BLOCK CARD ─────────────────────────────────────────────
    {"input": "intent: block-card | entities: card_type=debit | user: I lost my debit card please block it",
     "output": "I understand your concern. Please call our 24/7 support line immediately at 1-800-XXX-XXXX to block your debit card. Your security is our priority."},

    {"input": "intent: block-card | entities: card_type=credit | user: My credit card was stolen",
     "output": "We are sorry to hear that. Please call our fraud line immediately at 1-800-XXX-XXXX to block your credit card and dispute any unauthorized charges."},

    {"input": "intent: block-card | entities: card_type=debit | user: Someone used my card without permission",
     "output": "Please contact our fraud department immediately at 1-800-XXX-XXXX. Your card will be blocked and we will investigate any unauthorized transactions."},

    # ── TRANSACTION HISTORY ────────────────────────────────────
    {"input": "intent: transaction-history | entities: account=chequing, period=last month | user: Show me last month's transactions",
     "output": "You can view last month's transactions in the mobile app under Account History. Would you like help with anything else?"},

    {"input": "intent: transaction-history | entities: account=savings, period=last week | user: What transactions happened in my savings this week?",
     "output": "Your recent savings account transactions are available in the mobile app under Account History. Is there anything specific you are looking for?"},

    {"input": "intent: transaction-history | entities: account=chequing, period=last 3 months | user: Can I see my spending for the past 3 months?",
     "output": "Your last 3 months of transactions are available in the mobile app. You can also download a statement from the Documents section."},

    # ── ACCOUNT INFO ───────────────────────────────────────────
    {"input": "intent: account-info | entities: info_type=interest_rate | user: What is the interest rate on my savings account?",
     "output": "Our savings accounts currently offer a 2.5% annual interest rate. Would you like more details about our savings products?"},

    {"input": "intent: account-info | entities: info_type=account_number | user: What is my account number?",
     "output": "For security reasons, your full account number is only available in the mobile app under Account Details. Is there anything else I can help with?"},

    {"input": "intent: account-info | entities: info_type=branch | user: What branch am I with?",
     "output": "Your home branch details are available in the mobile app under Profile Settings. Would you like help finding a branch near you?"},

    # ── GENERAL FAQ ────────────────────────────────────────────
    {"input": "intent: general-faq | context: Branch hours are Monday to Friday 9am to 5pm | user: What are your branch hours?",
     "output": "Our branches are open Monday to Friday from 9am to 5pm. Some locations offer Saturday hours. Would you like to find your nearest branch?"},

    {"input": "intent: general-faq | context: ATMs are available 24/7 | user: Are your ATMs available on weekends?",
     "output": "Yes, our ATMs are available 24 hours a day, 7 days a week including weekends and holidays."},

    {"input": "intent: general-faq | context: Customer support is available 24/7 at 1-800-XXX-XXXX | user: How do I contact customer support?",
     "output": "You can reach our customer support team 24/7 by calling 1-800-XXX-XXXX or through the live chat in the mobile app."},

]

print(f"✅ Training dataset built — {len(training_data)} examples across 8 intents")

✅ Training dataset built — 24 examples across 8 intents


In [7]:
# Preview first and last example
print("FIRST EXAMPLE:")
print(f"INPUT:  {training_data[0]['input']}")
print(f"OUTPUT: {training_data[0]['output']}")

print("\nLAST EXAMPLE:")
print(f"INPUT:  {training_data[-1]['input']}")
print(f"OUTPUT: {training_data[-1]['output']}")

print(f"\nTotal examples: {len(training_data)}")

FIRST EXAMPLE:
INPUT:  intent: check-balance | entities: account=chequing | user: What is my chequing balance?
OUTPUT: Your chequing account balance is available in the mobile app or at any ATM. Is there anything else I can help you with?

LAST EXAMPLE:
INPUT:  intent: general-faq | context: Customer support is available 24/7 at 1-800-XXX-XXXX | user: How do I contact customer support?
OUTPUT: You can reach our customer support team 24/7 by calling 1-800-XXX-XXXX or through the live chat in the mobile app.

Total examples: 24


In [8]:
post_action_data = [

    # ── TRANSFER COMPLETED ─────────────────────────────────────
    {"input": "intent: transfer-complete | entities: amount=20000, from=chequing, to=email=john@email.com, new_balance=19000 | user: e-transfer $20,000 to john@email.com",
     "output": "Your e-transfer of $20,000 to john@email.com is complete. Your new chequing balance is $19,000. Would you like to make another transfer?"},

    {"input": "intent: transfer-complete | entities: amount=500, from=chequing, to=savings, new_balance=2300 | user: Move $500 to my savings",
     "output": "Done! $500 has been transferred to your savings account. Your new chequing balance is $2,300. Is there anything else I can help you with?"},

    {"input": "intent: transfer-complete | entities: amount=150, from=savings, to=chequing, new_balance=4850 | user: Transfer $150 to chequing",
     "output": "Transfer complete! $150 has been moved to your chequing account. Your new savings balance is $4,850. Would you like to do anything else?"},

    # ── BILL PAYMENT COMPLETED ─────────────────────────────────
    {"input": "intent: bill-paid | entities: payee=Rogers, amount=120, new_balance=3880 | user: Pay Rogers $120",
     "output": "Your Rogers bill of $120 has been paid successfully. Your new chequing balance is $3,880. Would you like to pay another bill?"},

    {"input": "intent: bill-paid | entities: payee=Hydro, amount=85, new_balance=2415 | user: Pay my hydro bill of $85",
     "output": "Done! Your Hydro bill of $85 has been paid. Your new balance is $2,415. Is there anything else I can help you with?"},

    {"input": "intent: bill-paid | entities: payee=Bell, amount=95, new_balance=1905 | user: Pay Bell $95",
     "output": "Your Bell payment of $95 was successful. Your new chequing balance is $1,905. Would you like to do anything else?"},

    # ── BALANCE REVEALED ───────────────────────────────────────
    {"input": "intent: balance-revealed | entities: account=chequing, balance=5240 | user: What is my chequing balance?",
     "output": "Your current chequing balance is $5,240. Would you like to make a transfer or pay a bill?"},

    {"input": "intent: balance-revealed | entities: account=savings, balance=12500 | user: How much is in my savings?",
     "output": "Your savings account balance is $12,500. Would you like to transfer funds or do something else?"},

    {"input": "intent: balance-revealed | entities: account=all, chequing=5240, savings=12500 | user: Show me all my balances",
     "output": "Here are your balances — Chequing: $5,240 | Savings: $12,500. Would you like to make a transaction?"},

    # ── CARD BLOCKED ───────────────────────────────────────────
    {"input": "intent: card-blocked | entities: card_type=debit | user: Block my debit card now",
     "output": "Your debit card has been successfully blocked. No further transactions can be made. A new card will be mailed to you within 5 business days."},

    {"input": "intent: card-blocked | entities: card_type=credit | user: Block my credit card it was stolen",
     "output": "Your credit card has been blocked immediately. Any unauthorized charges will be investigated. A replacement card will arrive within 5 business days."},

]

# Add to existing training data
training_data.extend(post_action_data)
print(f"✅ Dataset expanded — {len(training_data)} total examples")

✅ Dataset expanded — 35 total examples


In [9]:
# Full dataset summary
from collections import Counter

intents = [d['input'].split('|')[0].replace('intent:', '').strip() for d in training_data]
intent_counts = Counter(intents)

print(f"📊 Total examples: {len(training_data)}")
print(f"\n📋 Examples per intent:")
for intent, count in intent_counts.items():
    print(f"   {intent}: {count}")

📊 Total examples: 35

📋 Examples per intent:
   check-balance: 3
   transfer-funds: 3
   pay-bill: 3
   loan-inquiry: 3
   block-card: 3
   transaction-history: 3
   account-info: 3
   general-faq: 3
   transfer-complete: 3
   bill-paid: 3
   balance-revealed: 3
   card-blocked: 2


In [10]:
# Add one more card-blocked example to balance the dataset
training_data.append(
    {"input": "intent: card-blocked | entities: card_type=debit, reason=lost | user: I lost my debit card please block it immediately",
     "output": "Your debit card has been blocked successfully. No transactions can be made with this card. Please visit your nearest branch or call 1-800-XXX-XXXX to order a replacement card."}
)

# Verify
print(f"✅ Total examples: {len(training_data)}")
print(f"✅ card-blocked examples: {sum(1 for d in training_data if 'card-blocked' in d['input'])}")

✅ Total examples: 36
✅ card-blocked examples: 3


In [11]:
from datasets import Dataset

# Step 1 — Convert to HuggingFace Dataset
dataset = Dataset.from_list(training_data)
print(f"✅ Dataset created: {len(dataset)} examples")

# Step 2 — Define tokenization function
def preprocess(example):
    model_inputs = tokenizer(
        example["input"],
        max_length=256,
        padding="max_length",
        truncation=True
    )
    labels = tokenizer(
        example["output"],
        max_length=128,
        padding="max_length",
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Step 3 — Split 80% train, 20% test
split = dataset.train_test_split(test_size=0.2, seed=42)
train_data = split["train"]
test_data  = split["test"]

print(f"✅ Train examples: {len(train_data)}")
print(f"✅ Test examples:  {len(test_data)}")

# Step 4 — Tokenize both splits
tokenized_train = train_data.map(preprocess)
tokenized_test  = test_data.map(preprocess)

print("✅ Tokenization complete — ready to train!")

✅ Dataset created: 36 examples
✅ Train examples: 28
✅ Test examples:  8


Map:   0%|          | 0/28 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

✅ Tokenization complete — ready to train!


In [12]:
import torch

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"✅ Model running on: {device}")
print(f"✅ GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

✅ Model running on: cuda
✅ GPU name: Tesla T4


In [19]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import Dataset

# Reload model fresh
model_name = "google/flan-t5-base"
tokenizer  = T5Tokenizer.from_pretrained(model_name)
model      = T5ForConditionalGeneration.from_pretrained(model_name)
print("✅ Model reloaded fresh")

# Fix tokenization - remove extra columns
def preprocess(example):
    model_inputs = tokenizer(
        example["input"],
        max_length=256,
        padding="max_length",
        truncation=True
    )
    labels = tokenizer(
        example["output"],
        max_length=128,
        padding="max_length",
        truncation=True
    )
    model_inputs["labels"] = [
        -100 if token == tokenizer.pad_token_id else token
        for token in labels["input_ids"]
    ]
    return model_inputs

# Rebuild dataset and remove string columns
dataset    = Dataset.from_list(training_data)
split      = dataset.train_test_split(test_size=0.2, seed=42)

tokenized_train = split["train"].map(preprocess, remove_columns=["input", "output"])
tokenized_test  = split["test"].map(preprocess, remove_columns=["input", "output"])

# Set format to torch tensors
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")
print("✅ Dataset tokenized and formatted correctly")

# Training
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/models/response_generator",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=3e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    load_best_model_at_end=True,
    fp16=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer
)

print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Model reloaded fresh


Map:   0%|          | 0/28 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


✅ Training complete!


In [20]:
def generate_response(intent, entities, user_input, context=""):
    if context:
        prompt = f"intent: {intent} | entities: {entities} | context: {context} | user: {user_input}"
    else:
        prompt = f"intent: {intent} | entities: {entities} | user: {user_input}"

    inputs = tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test with real banking queries
print("=" * 60)
print("🏦 BANKING CHATBOT — MODEL TEST")
print("=" * 60)

print("\n1️⃣ CHECK BALANCE:")
print(generate_response("check-balance", "account=savings", "How much do I have in savings?"))

print("\n2️⃣ TRANSFER FUNDS:")
print(generate_response("transfer-funds", "amount=200, from=chequing, to=savings", "Transfer $200 to my savings"))

print("\n3️⃣ PAY BILL:")
print(generate_response("pay-bill", "payee=Rogers, amount=120", "Pay my Rogers bill"))

print("\n4️⃣ BLOCK CARD:")
print(generate_response("block-card", "card_type=credit", "My credit card was stolen"))

print("\n5️⃣ LOAN INQUIRY:")
print(generate_response("loan-inquiry", "loan_type=mortgage", "What are your mortgage rates?", "Mortgage rates start at 4.5%"))

print("\n6️⃣ TRANSFER COMPLETE:")
print(generate_response("transfer-complete", "amount=500, from=chequing, to=savings, new_balance=2300", "Move $500 to my savings"))

print("\n" + "=" * 60)
print("✅ Model test complete!")

🏦 BANKING CHATBOT — MODEL TEST

1️⃣ CHECK BALANCE:
Can you help?

2️⃣ TRANSFER FUNDS:
You can transfer $200 to your savings account.

3️⃣ PAY BILL:
You can pay your Rogers bill by logging into your account.

4️⃣ BLOCK CARD:
Your credit card has been blocked. You can no longer access your account.

5️⃣ LOAN INQUIRY:
You can start at 4.5%.

6️⃣ TRANSFER COMPLETE:
You can transfer $500 to your savings account. Can you help?

✅ Model test complete!


In [21]:
# Load the BASE model (before fine-tuning) for comparison
from transformers import T5ForConditionalGeneration, T5Tokenizer

base_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
base_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
base_model = base_model.to(device)

def zero_shot_response(user_input):
    inputs = base_tokenizer(
        user_input,
        return_tensors="pt",
        max_length=256,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = base_model.generate(**inputs, max_length=128, num_beams=4)
    return base_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test zero-shot on same queries
print("=" * 60)
print("🤖 ZERO-SHOT BASE MODEL (no fine-tuning)")
print("=" * 60)

print("\n1️⃣ CHECK BALANCE:")
print(zero_shot_response("How much do I have in my savings account?"))

print("\n2️⃣ TRANSFER FUNDS:")
print(zero_shot_response("Transfer $200 from chequing to savings"))

print("\n3️⃣ PAY BILL:")
print(zero_shot_response("Pay my Rogers bill of $120"))

print("\n4️⃣ BLOCK CARD:")
print(zero_shot_response("My credit card was stolen, block it"))

print("\n5️⃣ LOAN INQUIRY:")
print(zero_shot_response("What are your mortgage rates?"))

print("\n6️⃣ TRANSFER COMPLETE:")
print(zero_shot_response("Move $500 to my savings, show new balance"))

print("\n" + "=" * 60)
print("✅ Zero-shot test complete!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


🤖 ZERO-SHOT BASE MODEL (no fine-tuning)

1️⃣ CHECK BALANCE:
a few hundred thousand dollars

2️⃣ TRANSFER FUNDS:
Chequing: Transfer $200 from chequing to savings: Transfer $200 from chequing to savings: Transfer $200 from chequing to saving: Transfer $200 from chequing to saving: Transfer $200 from chequing to saving: Transfer $200 from chequing to saving: Transfer $200 from chequing to saving: Transfer $200 from chequing to saving: Transfer $200 from chequing to saving: Transfer $200 from chequing to saving: Transfer $200 from chequing to saving: Transfer $200 from chequing to saving: Transfer $200 from

3️⃣ PAY BILL:
I'm going to Roger's tonight. I'd like to pay $120. I'm going to Roger's tonight. I'm going to Roger's tonight. I'm going to Roger's tonight. I'm going to Roger's tonight. I'm going to Roger's tonight. I'm going to Roger's tonight. I'm going to Roger's tonight. I'm going to Roger's tonight. I'm going to Roger's tonight. I'm going to Roger's tonight. I'm going to Roger's



In [22]:
extra_data = [
    # CHECK BALANCE (more variety)
    {"input": "intent: check-balance | entities: account=chequing | user: Can you tell me my chequing balance?",
     "output": "Your chequing account balance is available in the mobile app or at any ATM. Would you like to do anything else?"},
    {"input": "intent: check-balance | entities: account=savings | user: I want to check my savings",
     "output": "Your savings account balance is available in the mobile app under Accounts. Is there anything else I can help you with?"},
    {"input": "intent: check-balance | entities: account=chequing | user: How much money do I have?",
     "output": "Your current chequing balance is available in the mobile app. Would you like to make a transfer or pay a bill?"},

    # TRANSFER FUNDS (more variety)
    {"input": "intent: transfer-funds | entities: amount=300, from=chequing, to=savings | user: I want to move $300 to savings",
     "output": "Your request to transfer $300 from chequing to savings is ready. Please confirm to complete the transfer."},
    {"input": "intent: transfer-funds | entities: amount=750, from=savings, to=chequing | user: Transfer $750 to my chequing please",
     "output": "I have received your request to transfer $750 from savings to chequing. Please confirm to proceed."},
    {"input": "intent: transfer-funds | entities: amount=50, from=chequing, to=Mary | user: Send $50 to Mary",
     "output": "Your request to send $50 to Mary is ready. Please confirm the recipient details and approve to proceed."},

    # PAY BILL (more variety)
    {"input": "intent: pay-bill | entities: payee=Netflix, amount=17 | user: Pay my Netflix subscription",
     "output": "Your request to pay $17 to Netflix is ready. Please confirm to complete the payment."},
    {"input": "intent: pay-bill | entities: payee=Insurance, amount=200 | user: Pay my insurance bill of $200",
     "output": "Your request to pay $200 to Insurance has been received. Please confirm to proceed with the payment."},
    {"input": "intent: pay-bill | entities: payee=Rent, amount=1500 | user: I need to pay my rent of $1500",
     "output": "Your request to pay $1,500 for rent is ready. Please confirm to complete the payment."},

    # TRANSFER COMPLETE (more variety)
    {"input": "intent: transfer-complete | entities: amount=300, from=chequing, to=savings, new_balance=1700 | user: Move $300 to savings",
     "output": "Done! $300 has been transferred to your savings account. Your new chequing balance is $1,700. Would you like to do anything else?"},
    {"input": "intent: transfer-complete | entities: amount=50, from=chequing, to=Mary, new_balance=950 | user: Send $50 to Mary",
     "output": "Your e-transfer of $50 to Mary is complete. Your new chequing balance is $950. Would you like to make another transfer?"},
    {"input": "intent: transfer-complete | entities: amount=750, from=savings, to=chequing, new_balance=3250 | user: Transfer $750 to chequing",
     "output": "Transfer complete! $750 has been moved to your chequing account. Your new savings balance is $3,250. Is there anything else I can help you with?"},

    # BILL PAID (more variety)
    {"input": "intent: bill-paid | entities: payee=Netflix, amount=17, new_balance=2983 | user: Pay Netflix $17",
     "output": "Your Netflix payment of $17 was successful. Your new chequing balance is $2,983. Would you like to pay another bill?"},
    {"input": "intent: bill-paid | entities: payee=Insurance, amount=200, new_balance=4800 | user: Pay insurance $200",
     "output": "Done! Your insurance payment of $200 has been processed. Your new balance is $4,800. Is there anything else I can help you with?"},
    {"input": "intent: bill-paid | entities: payee=Rent, amount=1500, new_balance=3500 | user: Pay rent $1500",
     "output": "Your rent payment of $1,500 was successful. Your new chequing balance is $3,500. Would you like to do anything else?"},

    # BALANCE REVEALED (more variety)
    {"input": "intent: balance-revealed | entities: account=chequing, balance=3200 | user: What is my balance?",
     "output": "Your current chequing balance is $3,200. Would you like to make a transfer or pay a bill?"},
    {"input": "intent: balance-revealed | entities: account=savings, balance=8750 | user: Check my savings balance",
     "output": "Your savings account balance is $8,750. Would you like to transfer funds or do something else?"},
    {"input": "intent: balance-revealed | entities: account=all, chequing=3200, savings=8750 | user: Show all my balances please",
     "output": "Here are your balances — Chequing: $3,200 | Savings: $8,750. Total: $11,950. Would you like to make a transaction?"},

    # LOAN INQUIRY (more variety)
    {"input": "intent: loan-inquiry | entities: loan_type=student | context: Student loans available from 5.5% | user: Do you offer student loans?",
     "output": "Yes, we offer student loans starting at 5.5%. Would you like to speak with a loan advisor for more details?"},
    {"input": "intent: loan-inquiry | entities: loan_type=business | context: Business loans from 7.2% | user: I need a business loan",
     "output": "We offer business loans starting at 7.2%. Would you like to book an appointment with our business banking team?"},
    {"input": "intent: loan-inquiry | entities: loan_type=personal, amount=10000 | context: Personal loans available from 6.9% | user: Can I get a $10,000 personal loan?",
     "output": "A $10,000 personal loan may be available starting at 6.9% APR depending on your credit profile. Would you like to apply?"},

    # GENERAL FAQ (more variety)
    {"input": "intent: general-faq | context: Online banking is available 24/7 | user: Can I do banking online?",
     "output": "Yes, our online banking is available 24/7. You can check balances, transfer funds, and pay bills anytime through our mobile app or website."},
    {"input": "intent: general-faq | context: New accounts can be opened online or in branch | user: How do I open a new account?",
     "output": "You can open a new account online through our website or by visiting any branch. Would you like help getting started?"},
    {"input": "intent: general-faq | context: Lost cards can be reported by calling 1-800-XXX-XXXX | user: What do I do if I lose my card?",
     "output": "If you lose your card, please call us immediately at 1-800-XXX-XXXX to report it and request a replacement. Your security is our priority."},
]

# Add to existing training data
training_data.extend(extra_data)
print(f"✅ Dataset expanded — {len(training_data)} total examples")

✅ Dataset expanded — 60 total examples


In [23]:
# Rebuild and retokenize with expanded dataset
dataset   = Dataset.from_list(training_data)
split     = dataset.train_test_split(test_size=0.2, seed=42)

tokenized_train = split["train"].map(preprocess, remove_columns=["input", "output"])
tokenized_test  = split["test"].map(preprocess, remove_columns=["input", "output"])

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

print(f"✅ Train examples: {len(split['train'])}")
print(f"✅ Test examples:  {len(split['test'])}")

# Reload fresh model
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
model = model.to(device)
print("✅ Fresh model loaded")

# Retrain
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/models/response_generator",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=3e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    load_best_model_at_end=True,
    fp16=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer
)

print("🚀 Retraining with 60 examples and 10 epochs...")
trainer.train()
print("✅ Retraining complete!")

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

✅ Train examples: 48
✅ Test examples:  12


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


✅ Retraining complete!


In [24]:
print("=" * 60)
print("🏦 BANKING CHATBOT — RETRAINED MODEL TEST")
print("=" * 60)

print("\n1️⃣ CHECK BALANCE:")
print(generate_response("check-balance", "account=savings", "How much do I have in savings?"))

print("\n2️⃣ TRANSFER FUNDS:")
print(generate_response("transfer-funds", "amount=200, from=chequing, to=savings", "Transfer $200 to my savings"))

print("\n3️⃣ PAY BILL:")
print(generate_response("pay-bill", "payee=Rogers, amount=120", "Pay my Rogers bill"))

print("\n4️⃣ BLOCK CARD:")
print(generate_response("block-card", "card_type=credit", "My credit card was stolen"))

print("\n5️⃣ LOAN INQUIRY:")
print(generate_response("loan-inquiry", "loan_type=mortgage", "What are your mortgage rates?", "Mortgage rates start at 4.5%"))

print("\n6️⃣ TRANSFER COMPLETE:")
print(generate_response("transfer-complete", "amount=500, from=chequing, to=savings, new_balance=2300", "Move $500 to my savings"))

print("\n" + "=" * 60)
print("✅ Test complete!")

🏦 BANKING CHATBOT — RETRAINED MODEL TEST

1️⃣ CHECK BALANCE:
Your savings account balance is in the account. Would you like to make a payment?

2️⃣ TRANSFER FUNDS:
Your chequing chequing account is in the chequing account. Would you like to make a transfer?

3️⃣ PAY BILL:
You can make a request to make a payment, to make a payment, or to pay a bill.

4️⃣ BLOCK CARD:
You can make a credit card, in a card, . You can make a transfer.

5️⃣ LOAN INQUIRY:
You can make a request to make a loan.

6️⃣ TRANSFER COMPLETE:
Your chequing chequing balance is $2,300. Would you like to make a transfer?

✅ Test complete!


In [25]:
import evaluate

bleu = evaluate.load("bleu")

# Generate predictions on test set
predictions = []
references  = []

for example in split["test"]:
    # Get model prediction
    inputs = tokenizer(
        example["input"],
        return_tensors="pt",
        max_length=256,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_length=128, num_beams=4)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

    predictions.append(pred)
    references.append([example["output"]])

# Calculate BLEU
results = bleu.compute(predictions=predictions, references=references)
print(f"✅ Fine-tuned BLEU Score: {results['bleu']:.4f}")

# Zero-shot BLEU
zero_predictions = []
for example in split["test"]:
    inputs = base_tokenizer(
        example["input"],
        return_tensors="pt",
        max_length=256,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = base_model.generate(**inputs, max_length=128, num_beams=4)
    pred = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
    zero_predictions.append(pred)

zero_results = bleu.compute(predictions=zero_predictions, references=references)
print(f"✅ Zero-shot BLEU Score:  {zero_results['bleu']:.4f}")

print(f"\n📊 Improvement: {((results['bleu'] - zero_results['bleu']) / zero_results['bleu'] * 100):.1f}% better than zero-shot")

✅ Fine-tuned BLEU Score: 0.1063
✅ Zero-shot BLEU Score:  0.0430

📊 Improvement: 147.0% better than zero-shot


In [28]:
# Clear ALL models from GPU
import torch
import gc

# Delete all models
try:
    del model
except: pass
try:
    del model_run3
except: pass
try:
    del base_model
except: pass

# Force clear GPU
gc.collect()
torch.cuda.empty_cache()

# Verify memory is free
print(f"✅ GPU memory cleared")
print(f"✅ Free GPU memory: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

✅ GPU memory cleared
✅ Free GPU memory: 5.89 GB


In [30]:
import shutil
import os

# Delete old checkpoints to free up space
folders_to_delete = [
    "/content/models/response_generator",
    "/content/models/response_generator_run3"
]

for folder in folders_to_delete:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"🗑️ Deleted: {folder}")

# Check disk space after cleanup
import shutil
total, used, free = shutil.disk_usage("/content")
print(f"\n✅ Free disk space: {free / 1e9:.2f} GB")
print(f"✅ Used disk space: {used / 1e9:.2f} GB")

🗑️ Deleted: /content/models/response_generator
🗑️ Deleted: /content/models/response_generator_run3

✅ Free disk space: 73.90 GB
✅ Used disk space: 47.03 GB


In [31]:
# Reload fresh model for Run 3
model_run3 = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
model_run3 = model_run3.to(device)
print(f"✅ Fresh model loaded")

# Run 3 — lower learning rate, save only at the end
training_args_run3 = Seq2SeqTrainingArguments(
    output_dir="/content/models/response_generator_run3",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=1e-4,                   # ← lower than Run 2
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="no",                   # ← don't save every epoch
    predict_with_generate=True,
    fp16=False
)

trainer_run3 = Seq2SeqTrainer(
    model=model_run3,
    args=training_args_run3,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer
)

print("🚀 Starting Run 3...")
trainer_run3.train()
print("✅ Run 3 complete!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Fresh model loaded
🚀 Starting Run 3...


Epoch,Training Loss,Validation Loss
1,9.774772,9.646343
2,9.487801,8.967492
3,9.017312,8.592254
4,8.773926,8.129064
5,8.378775,7.750960
6,8.229639,7.488163
7,8.041571,7.336070
8,7.841712,7.216725
9,7.788791,7.100343
10,7.668325,7.103262


✅ Run 3 complete!


In [32]:
bleu = evaluate.load("bleu")

# Generate predictions using Run 3 model
predictions_run3 = []
references = []

for example in split["test"]:
    inputs = tokenizer(
        example["input"],
        return_tensors="pt",
        max_length=256,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model_run3.generate(**inputs, max_length=128, num_beams=4)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions_run3.append(pred)
    references.append([example["output"]])

results_run3 = bleu.compute(predictions=predictions_run3, references=references)
print(f"✅ Run 3 BLEU Score: {results_run3['bleu']:.4f}")
print(f"✅ Run 2 BLEU Score: 0.1063")
print(f"✅ Zero-shot BLEU:   0.0430")

✅ Run 3 BLEU Score: 0.0882
✅ Run 2 BLEU Score: 0.1063
✅ Zero-shot BLEU:   0.0430


In [34]:
import os

# Reload and retrain Run 2 (our best model)
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
model = model.to(device)
print("✅ Fresh model loaded")

training_args_run2 = Seq2SeqTrainingArguments(
    output_dir="/content/models/best_model",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=3e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="no",              # ← no disk saving during training
    predict_with_generate=True,
    fp16=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args_run2,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer
)

print("🚀 Retraining Run 2 (best model)...")
trainer.train()
print("✅ Done! Now saving...")

# Save immediately after training
import os
save_path = "/content/models/best_model"
os.makedirs(save_path, exist_ok=True)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Best model saved to {save_path}")
print(f"✅ Files: {os.listdir(save_path)}")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Fresh model loaded
🚀 Retraining Run 2 (best model)...


Epoch,Training Loss,Validation Loss
1,9.716240,9.066592
2,9.065143,8.498210
3,8.504415,7.794069
4,8.061995,7.324259
5,7.527512,6.801664
6,7.287359,6.535437
7,6.985053,6.366307
8,6.686120,6.100687
9,6.593550,5.921937
10,6.333537,5.885421


✅ Done! Now saving...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Best model saved to /content/models/best_model
✅ Files: ['model.safetensors', 'tokenizer.json', 'generation_config.json', 'config.json', 'tokenizer_config.json']


In [35]:
import shutil

# Zip the model folder
shutil.make_archive("/content/best_model", "zip", "/content/models/best_model")
print("✅ Model zipped!")

# Download it
from google.colab import files
files.download("/content/best_model.zip")
print("✅ Download started!")

✅ Model zipped!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!
